Модель №1 Для начала посчитаем ошибку на Кросс-Валидации и Тесте для нашей самой базовой модели до вычленения каких-либо признаков, а просто взяв все вещественные колонки.
Как и в предыдущих уроках, будем последовательно выполнять задания в прикрепленном ноутбуке. Также нам понадобятся данные - initial_data.csv.

Выполните код в ноутбуке. В качестве ответа у вас должно получиться два числа: среднее качество моделей Кросс-Валидации на соответствующих валидационных выборках и качество модели, обученной на полной тренировочной выборке, на изначальной тестовой выборке. Введите их через запятую и пробел, в качестве разделителя используйте точку, ответ округлите до тысячных.

In [2]:
import pandas as pd

In [3]:
initial_data = pd.read_csv('initial_data.csv', index_col='id')

initial_cols = ['vendor_id', 'passenger_count', 'pickup_longitude',
                'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude',
                'trip_duration']

initial_data = initial_data[initial_cols]

In [ ]:
# Замерить MSLE. Можно показать, что для оптимизации MSLE, достаточно логарифмировать таргетную переменную, а потом оптимизировать привычные MSE
import numpy as np

initial_data = initial_data.assign(log_trip_duration=np.log1p(initial_data['trip_duration']))
initial_data = initial_data.drop('trip_duration', axis=1)

In [ ]:
# Выделить test

from sklearn.model_selection import train_test_split

X = initial_data.drop('log_trip_duration', axis=1)
y = initial_data['log_trip_duration']

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=42)

In [ ]:
# Установить метод кросс-валидации

from sklearn.model_selection import KFold

splitter = KFold(n_splits=20, shuffle=True, random_state=33)

In [ ]:
# Воспользуемся на кросс-валидации методом cross_validate

from sklearn.model_selection import cross_validate
from sklearn.linear_model import LinearRegression

model = LinearRegression()

cv_result = cross_validate(model, X_train, y_train, 
                           scoring='neg_mean_squared_error',
                           cv=splitter, return_train_score=True)

In [ ]:
# Взять среднее по валидации и получить качество на кросс-валидации

np.mean(-cv_result['test_score'])

0.6134219105328517

In [ ]:
# Построить модель на всей тренировочной выборке и замерить качество на тесте - финальный замер

model = LinearRegression()

model.fit(X_train, y_train)

model_predictions = model.predict(X_test)

np.mean(((model_predictions - y_test))**2)

0.6056668879526188

Модель №2. Проделаем все то же самое, только для модели с более осознанными признаками, которые удалось получить ранее. Данные лежат в файле processed_data.csv.

Выполните код в ноутбуке. В качестве ответа у вас должно получиться два числа: среднее качество моделей Кросс-Валидации на соответствующих валидационных выборках и качество модели, обученной на полной тренировочной выборке, на изначальной тестовой выборке. Введите их через запятую и пробел, качестве разделителя используйте точку, ответ округлите до тысячных.

Пример ответа:

0.653, 0.602

In [16]:
processed_data = pd.read_csv('processed_data.csv', index_col='id')

In [17]:
processed_data.head()

,vendor_id,passenger_count,store_and_fwd_flag,trip_duration,distance_km
id,,,,,
id2875421,1,930.399753,0,455.0,1.500479
id2377394,0,930.399753,0,663.0,1.807119
id3858529,1,930.399753,0,2124.0,6.392080
id3504673,1,930.399753,0,429.0,1.487155
id2181028,1,930.399753,0,435.0,1.189925


In [ ]:
# Замерить RMSLE. Можно показать, что для оптимизации RMSLE, достаточно логарифмировать таргетную переменную, а потом оптимизировать привычные MSE

import numpy as np

processed_data = processed_data.assign(log_trip_duration=np.log1p(processed_data['trip_duration']))
processed_data = processed_data.drop('trip_duration', axis=1)

In [19]:
X_2 = processed_data.drop('log_trip_duration', axis=1)
y_2 = processed_data['log_trip_duration']

In [ ]:
# Можно воспользоваться старым random_state и аналогично через train_test_split поделить выборку. Или можно зафиксировать индексы в разделенных данных

test_indexes = X_test.index
train_indexes = X_train.index

In [ ]:
# и применить индексы на новых

X_train_2 = X_2[X_2.index.isin(train_indexes)]
y_train_2 = y_2[y_2.index.isin(train_indexes)]

X_test_2 = X_2[X_2.index.isin(test_indexes)]
y_test_2 = y_2[y_2.index.isin(test_indexes)]

In [ ]:
# Запустить валидацию

cv_result_2 = cross_validate(model, X_train_2, y_train_2, 
                             scoring='neg_mean_squared_error',
                             cv=splitter, return_train_score=True)

In [23]:
np.mean(-cv_result_2['test_score'])

0.4313257704232939

In [ ]:
# Теперь построим модель и замерим качество на тесте

model_2 = LinearRegression()

model_2.fit(X_train_2, y_train_2)

model_predictions_2 = model_2.predict(X_test_2)

np.mean(((model_predictions_2 - y_test_2))**2)

0.4074669546810228